# When Nvidia Moves, Who Follows?

## Is AI exposure visible on paper or in stock prices?

### Research Question

Do firms identified as AI-exposed through their sector, R&D spending,
or membership in thematic ETFs also display greater sensitivity to
Nvidia's earnings announcements?

Does this relationship extend to international equity markets?

### Research Period
2018–2026
### Authors
Estelle Aoun & Aya Rhazi

## 1.Hypotheses and research design
**H1:** Firms' abnormal returns have very different sensitivities to Nvidia. A minority responds especially strongly

**H2:** Sector, R&D/revenue and thematic ETF labels are associated with these sensitivities, conditional on the available controls

**H3:** Sensitivities are higher after 30 November 2022 (ChatGPT's launch)

**H4:** Country-level exposure differs across stock markets and correlates with technological specialization and semiconductor trade links

An announcement occurring **after 16:00 US Eastern** is assigned to the **following US trading session**: the reaction is the close-to-close return from the announcement day's close to that next close. This is appropriate for US equities and the **US-listed country ETFs** used below. It is **not** a local-exchange-hours event study.

We use an estimation window of $[-250,-30]$ trading days to capture roughly one full trading year of baseline market behavior while stopping 30 days ahead of the release to prevent pre-announcement drift or information leaks from biasing our benchmark.
For each asset $i$ and earnings event $e$, estimate the market model over trading sessions $[-250,-30]$ relative to the first post-announcement session :

$$R_{i,t}=\alpha_{i,e}+\beta_{i,e}R_{m,t}+\varepsilon_{i,t}.$$

Then measure abnormal returns at the event:

$$AR_{i,e}=R_{i,e}-(\hat\alpha_{i,e}+\hat\beta_{i,e}R_{m,e}).$$

Finally, across earnings announcements, estimate the **revealed exposure** slope:

$$AR_{i,e}=a_i+b_i AR_{\mathrm{NVDA},e}+u_{i,e}.$$

A positive $b_i$ indicates directional co-movement; it is **not** a causal effect and is **not** the firm's ordinary market beta.

In [2]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, IFrame

#Ensure the project root is in sys.path so our custom 'src' modules import cleanly
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

#Import custom data-loading, econometric and visualization functions from src/
from src.settings import Settings, CONFIG, PROCESSED, FIGURES
from src.analysis import (
    load_prices, check_events, event_session, data_quality, event_study,
    attach_nvda_shocks, sensitivity_by_stock, pre_post_sensitivity,
    market_betas_before_cutoff, cross_section_regression, country_regression,
    yearly_pca, placebo_nvda_shocks,
)
from src.plots import (
    plot_nvda_reactions, plot_sensitivity_distribution, plot_pre_post,
    plot_pca, plot_country_map, plot_country_two_periods,
)

#Initialize project paths and make sure output directories exist
settings = Settings()
DATA_ROOT = Path(os.environ.get("NVIDIA_PROJECT_DATA_DIR", ROOT / "data" / "raw"))
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

#Configure pandas display so summary tables show enough columns cleanly
pd.set_option("display.max_columns", 15)
print("Working with data from:", DATA_ROOT)

Working with data from: c:\Users\alexi\ai-exposure-financial-markets\data\raw


## 2. Data collection, cleaning and quality checks

**Data pipeline and sources:** We combine multi-source public financial datasets stored as raw CSV snapshots in `data/raw/`, which are subsequently validated, aligned on US trading calendars, and joined by ticker and date. Baseline equity prices (Nvidia, Nasdaq-100 constituents, `SPY`/`QQQ` benchmarks, and 29 US-listed country ETFs) and earnings dates are collected via `python -m src.collect`. Firm-level R&D expenditures, revenues, and thematic ETF memberships are retrieved separately via `python -m src.firm_data` (using a compliant `SEC_USER_AGENT` header for SEC EDGAR access). Throughout the cleaning pipeline, missing observations are explicitly preserved as `NaN` rather than imputed with zeros, and missing ETF files are never treated as evidence of non-membership.

**Reproducibility:** All raw snapshots, automated validation checks, and collection scripts are versioned and documented in `README.md` so the full dataset can be regenerated from scratch.

In [5]:
required = ["us_prices.csv", "country_etf_prices.csv", "nasdaq100_constituents.csv"]
missing = [filename for filename in required if not (DATA_ROOT / filename).exists()]
if missing:
    raise FileNotFoundError(
        "Missing source snapshots: " + ", ".join(missing) + ". "
        "From the project root, run 'python -m src.collect' on a machine with internet access."
    )

us_prices = load_prices(DATA_ROOT / "us_prices.csv")
country_prices = load_prices(DATA_ROOT / "country_etf_prices.csv")
constituents = pd.read_csv(DATA_ROOT / "nasdaq100_constituents.csv")
events = check_events(pd.read_csv(CONFIG / "earnings_nvda.csv"))
countries = pd.read_csv(CONFIG / "country_etfs.csv")

assert {"NVDA", "SPY", "QQQ"}.issubset(us_prices.columns), "Missing benchmark or Nvidia prices"
assert len(events) == 36, "Review the event list before changing the study period"

firm_tickers = sorted(set(constituents.ticker) & set(us_prices.columns) - {"NVDA", "SPY", "QQQ"})
country_tickers = sorted(set(countries.etf) & set(country_prices.columns))
print(f"{len(firm_tickers)} firm tickers, {len(country_tickers)} country ETFs, {len(events)} earnings dates")

quality = data_quality(us_prices, events)
display(pd.DataFrame(quality.items(), columns=["Data-quality indicator", "Value"]))
print("First event:", events.announcement_date.min().date(),
      "| last event:", events.announcement_date.max().date())
print("First after-close release -> reaction session:",
      event_session(events.announcement_date.min(), us_prices.index[us_prices.SPY.notna()]).date())

100 firm tickers, 29 country ETFs, 36 earnings dates


,Data-quality indicator,Value
0,assets,103.00000
1,trading_days,2209.00000
2,announcement_dates,36.00000
3,price_observations_missing_pct,8.46405
4,stocks_with_80pct_coverage,89.00000
5,benchmark_missing_pct,0.00000


First event: 2018-02-08 | last event: 2026-11-17
First after-close release -> reaction session: 2018-02-09


In [6]:
# Missing-price coverage must be reported, not silently imputed.
coverage = us_prices.loc[us_prices.index >= "2018-01-01"].notna().mean().sort_values()
display(coverage.head(12).rename("observed_share").to_frame())

# Keep issuer sector classifications explicitly labelled as ICB, not GICS.
if (DATA_ROOT / "firm_labels.csv").exists():
    labels = pd.read_csv(DATA_ROOT / "firm_labels.csv")
    print("SEC/holdings snapshot found.")
else:
    print("SEC/ETF labels not yet collected: sector-only associations will remain preliminary.")
    labels = constituents.copy()
    labels["technology_label"] = labels.sector.str.contains("technology", case=False, na=False).astype(int)
    labels["semiconductor_label"] = labels.subsector.str.contains("semiconductor", case=False, na=False).astype(int)
    for name in ["rd_revenue", "ai_etf_member", "semi_etf_member", "market_cap_usd"]:
        labels[name] = np.nan

label_summary = labels[[c for c in ["technology_label", "semiconductor_label", "rd_revenue",
                                    "ai_etf_member", "semi_etf_member", "market_cap_usd"] if c in labels]].notna().sum()
display(label_summary.rename("non-missing firms").to_frame())

,observed_share
HONA,0.050112
SPCX,0.050559
CRWV,0.186130
SNDK,0.199553
NBIS,0.234452
ALAB,0.300671
ARM,0.358389
GEHC,0.441611
CEG,0.544072
APP,0.630425


SEC/holdings snapshot found.


,non-missing firms
technology_label,101
semiconductor_label,101
rd_revenue,58
ai_etf_member,0
semi_etf_member,0
market_cap_usd,0


### Interpretation of Data Coverage and Quality

* **Panel completeness and benchmark integrity:** Our dataset covers **103 US assets** (100 Nasdaq-100 firms plus `NVDA`, `SPY`, and `QQQ`) across **2,209 trading days** and **36 Nvidia quarterly earnings announcements** spanning from February 8, 2018 (with its first post-close market reaction measured on the February 9 trading session) through November 17, 2026. Crucially, the market benchmarks (`SPY` and `QQQ`) have **0.00% missing observations**, ensuring an uninterrupted baseline for estimating abnormal returns.
* **Missing price observations and survivorship bias:** Across the full equity panel, **8.46%** of price observations are missing, while **89 out of 100 firms** achieve at least 80% trading-day coverage. The lowest-coverage tickers—such as `HONA` (5.0%), `SPCX` (5.1%), `CRWV` (18.6%), `SNDK` (20.0%), and `ARM` (35.8%)—reflect companies that went public or were listed later in the 2018–2026 period rather than data collection failures. Because our constituent list is drawn from a current Nasdaq-100 snapshot rather than a historical point-in-time reconstruction, our findings describe how today's major firms reacted over their available trading histories and are subject to standard survivorship bias.
* **Firm-level explanatory variables and expected zeros:** Sector and subsector classifications (`technology_label` and `semiconductor_label`) are available for all **101 firms**, while direct SEC EDGAR filing extraction yields valid R&D-to-revenue intensity (`rd_revenue`) for **58 out of 101 firms** (the remaining 43 firms either do not separately report R&D expenses—such as financial or retail companies—or lack standardized XBRL tags). Meanwhile, `ai_etf_member`, `semi_etf_member`, and `market_cap_usd` show **0 non-missing firms** because external ETF issuer holdings CSVs were not manually added to `config/holdings/` and market capitalization is not extracted by the baseline SEC script. Consequently, our cross-sectional tests in **H2** will focus on sector classifications and SEC R&D intensity rather than thematic ETF membership or size controls[cite: 14, 15, 16, 18].

### H1 - Revealed exposure differs across firms


## 4. Do the AI labels predict sensitivity ? (H2)

## 5. Did the pattern change after ChatFPT? (H3)

### Descriptive PCA: are stocks moving more together ?

## 6. Gerography of the AI trade (H4)

### Adding country characteristics (World Bank and UN Comtrade)

## 7. Robustness checks

## 8. Synthesis and limits